# Split Documents into Retrieval Chunks

**Course author:** Jayneel Girnara

Prepare long text and loaded documents for retrieval by dividing them into smaller, overlapping chunks.

## Learning goals
- Split plain text with `RecursiveCharacterTextSplitter`.
- Understand the trade-off between chunk size and overlap.
- Split LangChain `Document` objects while preserving their metadata.

## Before you start
Install the project dependencies and ensure the sample text and PDF files are present under `data/`. This notebook demonstrates splitting locally and does not require a model API key.

## How to work through it
Run cells top to bottom. First inspect chunks from a short text example, then compare the text-loader and PDF-loader results. Notice how smaller chunks aid retrieval while overlap preserves context at boundaries.

## 1. Split a plain-text example

A **text splitter** divides a long string into smaller pieces. `chunk_size` is a character limit here, not a token count. `chunk_overlap` repeats some characters from the previous chunk to preserve continuity at boundaries. The toy text makes the boundary behavior visible.

**Checkpoint:** What can be lost if overlap is zero?

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent

load_dotenv(project_root / ".env")


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
 

In [3]:
texts="Today, I want to speak from the heart about three pillars that define the future of our nation\
our youth, our education system, and our farmers."

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=50,chunk_overlap=20)

In [5]:
chunks=splitter.split_text(texts)

In [7]:
chunks

['Today, I want to speak from the heart about three',
 'heart about three pillars that define the future',
 'define the future of our nationour youth, our',
 'youth, our education system, and our farmers.']

## 2. Split loaded documents

Split text and PDF `Document` objects while retaining their metadata. Chunk size controls the maximum text length; overlap keeps boundary context available.

`split_documents` accepts `Document` objects and carries their metadata into each resulting chunk. The right chunk size depends on the source, model context, and retrieval task; overlap cannot replace good chunk boundaries. This lesson prepares text for [Lesson 12](12_vector_embeddings.ipynb), where chunks are represented as vectors.

**Checkpoint:** Why keep source metadata on every chunk?

In [16]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [22]:
loader=TextLoader(project_root / "data" / "speech.txt")
texts=loader.load()

In [23]:
pdfloader=PyPDFLoader(project_root / "data" / "medical_report.pdf")
pdfs=pdfloader.load()

In [24]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)


In [26]:
text_chunks=splitter.split_documents(texts)
len(text_chunks)

2

In [27]:
text_chunks

[Document(metadata={'source': '../data/speech.txt'}, page_content='“My dear friends, students, and respected farmers,\n\nToday, I want to speak from the heart about three pillars that define the future of our nation:\nour youth, our education system, and our farmers.\n\n🎓 For the Youth\n\nYou are the energy that drives our state forward.\nOur commitment is to give you:\n\t•\tSkill-based education\n\t•\tDigital learning opportunities\n\t•\tBetter employment pathways\n\t•\tSupport for innovation, startups, and entrepreneurship\n\nYour dreams are not just your own—they are the dreams of our entire society.\n\n📚 For Education\n\nEducation must not be just a formality; it must be a force of transformation.\nWe aim to:\n\t•\tUpgrade government schools with smart classrooms\n\t•\tImprove teacher training\n\t•\tExpand scholarships for higher education\n\t•\tPromote technical and vocational courses\n\t•\tEnsure every child gets a fair opportunity to learn and grow\n\nEducation is the foundation

In [28]:
pdf_chunks=splitter.split_documents(pdfs)
len(pdf_chunks)

26